## Tipos de interés, descuento, rentas y VAN/TIR

Partimos del dataframe que sale del notebook `valor_presente`, que ya trae el `Sistema` de cada cliente y el valor de sus cuotas pendientes (`VA_hoy`, `VA_mitad`, `Valor_Final`).

In [ ]:
import pandas as pd
import numpy as np
import numpy_financial as npf

In [ ]:
df = pd.read_csv("Data/df_negocio_valor_presente.csv")
datos = df.dropna(subset=["Monto_Inicial", "Ratio_Interes", "Duracion"])

df[["ID", "Monto_Inicial", "Ratio_Interes", "Duracion", "Sistema", "VA_hoy"]].head()

### Tablas de amortización y rentas

Las mismas funciones que en `valor_presente`: hacen falta para sacar las cuotas (flujos del VAN) y para los ejemplos de rentas.

In [ ]:
def amortizacion_alemana(capital, interes_anual, plazo_meses):
    i = interes_anual / 100 / 12
    A = capital / plazo_meses

    tabla = []
    saldo = capital

    for t in range(1, plazo_meses + 1):
        interes = saldo * i
        cuota = A + interes
        saldo -= A

        tabla.append({
            "Mes": t,
            "Amortizacion": round(A, 2),
            "Interes": round(interes, 2),
            "Cuota": round(cuota, 2),
            "Saldo_restante": round(max(saldo, 0), 2)
        })

    return pd.DataFrame(tabla)


def amortizacion_francesa(capital, interes_anual, plazo_meses):
    i = interes_anual / 100 / 12
    cuota = capital * i / (1 - (1 + i) ** -plazo_meses)

    tabla = []
    saldo = capital

    for t in range(1, plazo_meses + 1):
        interes = saldo * i
        amortizacion = cuota - interes
        saldo -= amortizacion

        tabla.append({
            "Mes": t,
            "Amortizacion": round(amortizacion, 2),
            "Interes": round(interes, 2),
            "Cuota": round(cuota, 2),
            "Saldo_restante": round(max(saldo, 0), 2)
        })

    return pd.DataFrame(tabla)


def tabla(f):
    if f["Sistema"] == "aleman":
        return amortizacion_alemana(f["Monto_Inicial"], f["Ratio_Interes"], int(f["Duracion"]))
    return amortizacion_francesa(f["Monto_Inicial"], f["Ratio_Interes"], int(f["Duracion"]))

In [ ]:
def valor_renta(M, i, n=None, q=1, tipo="pospagable", m=0):
    if n is None:
        va = M / (1 + i - q)
    elif q == 1 + i:
        va = n * M / (1 + i)
    else:
        va = M * (1 - q ** n * (1 + i) ** -n) / (1 + i - q)

    if tipo == "prepagable":
        va = va * (1 + i)

    if n is None:
        vf = np.nan
    else:
        vf = va * (1 + i) ** n

    va = va / (1 + i) ** m
    return va, vf


def valor_flujos(flujos, i, m=0):
    flujos = np.array(flujos, dtype=float)
    t = np.arange(1, len(flujos) + 1)
    va = (flujos / (1 + i) ** t).sum()
    vf = va * (1 + i) ** len(flujos)
    return va / (1 + i) ** m, vf

### Tipos de interés

El `Ratio_Interes` del dataset es un TIN, es decir, un tipo nominal anual capitalizable mensualmente j(12). El tipo mensual es i(12) = j(12) / 12 y el tipo anual efectivo (TAE) sale de (1 + i(12))^12 = 1 + i. Con capitalización simple la TAE coincidiría con el TIN, con compuesta es algo mayor.

In [ ]:
df["Interes_mensual"] = df["Ratio_Interes"] / 100 / 12
df["TAE"] = (((1 + df["Interes_mensual"]) ** 12 - 1) * 100).round(4)

df[["ID", "Ratio_Interes", "Interes_mensual", "TAE"]].head()

### Actualización y descuento

Valor hoy de la última cuota del primer cliente con los cuatro tipos de descuento. El racional da siempre más valor que el comercial.

In [ ]:
def descuento(Cn, tasa, n, tipo):
    if tipo == "racional simple":
        return Cn / (1 + n * tasa)
    if tipo == "racional compuesto":
        return Cn * (1 + tasa) ** -n
    if tipo == "comercial simple":
        return Cn * (1 - n * tasa)
    return Cn * (1 - tasa) ** n


f = datos.iloc[0]
i = f["Ratio_Interes"] / 100 / 12
n = int(f["Duracion"])
cuota_final = tabla(f)["Cuota"].iloc[-1]

tipos = ["racional simple", "racional compuesto", "comercial simple", "comercial compuesto"]
pd.DataFrame({
    "Tipo": tipos,
    "Valor actual": [round(descuento(cuota_final, i, n, t), 2) for t in tipos]
})

Ejemplo de cada tipo de renta con el primer cliente, usando como término su cuota francesa. La constante pospagable y la variable con las cuotas alemanas tienen que dar el capital prestado.

In [ ]:
C = f["Monto_Inicial"]
M = C * i / (1 - (1 + i) ** -n)
cuotas_alemanas = amortizacion_alemana(C, f["Ratio_Interes"], n)["Cuota"]

casos = {
    "Constante temporal pospagable (vencida)": valor_renta(M, i, n),
    "Constante temporal prepagable (anticipada)": valor_renta(M, i, n, tipo="prepagable"),
    "Constante diferida 6 meses": valor_renta(M, i, n, m=6),
    "Constante perpetua": valor_renta(M, i),
    "Geométrica temporal (q = 1,001)": valor_renta(M, i, n, q=1.001),
    "Geométrica perpetua (q = 1,001)": valor_renta(M, i, q=1.001),
    "Variable temporal (cuotas alemanas)": valor_flujos(cuotas_alemanas, i),
}

print("Capital prestado:", round(C, 2))
pd.DataFrame(casos, index=["Valor actual", "Valor final"]).T.round(2)

### VAN y TIR

Desde el punto de vista del banco cada préstamo es una inversión: presta el capital hoy (-A) y cobra las cuotas después (Q1, ..., Qn). Los flujos se descuentan a la rentabilidad mínima exigida k, que es anual y se pasa a mensual con (1 + k)^(1/12) - 1. El préstamo es viable si VAN > 0, que es lo mismo que TIR > k. El índice de rentabilidad IR = VAN / A sirve para ordenar préstamos si el capital disponible es limitado.

In [ ]:
K_ANUAL = 5


def flujos_banco(f):
    return [-f["Monto_Inicial"]] + list(tabla(f)["Cuota"])


# Los flujos de cada préstamo se calculan una sola vez y se reutilizan
# para el VAN, la TIR y los escenarios.
flujos = datos.apply(flujos_banco, axis=1)


def van(fl, k_anual=K_ANUAL):
    k = (1 + k_anual / 100) ** (1 / 12) - 1
    return round(npf.npv(k, fl), 2)


def tir(fl):
    tir_mensual = npf.irr(fl)
    return round(((1 + tir_mensual) ** 12 - 1) * 100, 2)


df.loc[datos.index, "VAN"] = flujos.apply(van)
df.loc[datos.index, "TIR"] = flujos.apply(tir)
df["Viable"] = df["VAN"] > 0
df["IR"] = (df["VAN"] / df["Monto_Inicial"]).round(4)

df[["ID", "Sistema", "Monto_Inicial", "TAE", "VAN", "TIR", "Viable", "IR"]].head(10)

Resumen por sistema y número de préstamos viables. La TIR anual de cada préstamo tiene que salir prácticamente igual que su TAE.

In [ ]:
print(df.loc[datos.index, "Viable"].value_counts(), "\n")
print("Diferencia media entre TIR y TAE:", round((df["TIR"] - df["TAE"]).abs().mean(), 4))

df.groupby("Sistema")[["VAN", "TIR", "IR"]].mean().round(2)

Escenarios: VAN total de la cartera y préstamos viables según la rentabilidad exigida k.

In [ ]:
escenarios = []

for k in [3, 5, 7, 9]:
    van_k = flujos.apply(van, k_anual=k)
    escenarios.append({
        "k (%)": k,
        "VAN total": round(van_k.sum(), 2),
        "% préstamos viables": round((van_k > 0).mean() * 100, 1)
    })

pd.DataFrame(escenarios)

### Guardado del dataframe

In [ ]:
df.to_csv("Data/df_negocio_van_tir.csv", index=False)